# HealthCareMagic-100K Dataset Exploration

This notebook explores the processed HealthCareMagic-100K dataset for fine-tuning LLaMA 3.1 8B model.

The main focus areas are:
1. Dataset statistics and distribution
2. Query and response length analysis
3. Medical specialty identification
4. Common medical conditions and treatments
5. Patient demographics (age, gender, etc.)

In [1]:
import os
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path
from tqdm.notebook import tqdm
import spacy
import warnings
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Ignore warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('ggplot')
sns.set(style="whitegrid")

# Create a directory to save visualization results
os.makedirs('visualization_results', exist_ok=True)

In [2]:
# Load the processed dataset
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Path to processed dataset
processed_data_path = "../data/processed/healthcare_magic_instructions.jsonl"

# Load the processed dataset
try:
    dataset = load_jsonl(processed_data_path)
    print(f"Loaded {len(dataset)} examples from {processed_data_path}")
except FileNotFoundError:
    print(f"File not found: {processed_data_path}")
    print("Please run the data processing script first or update the path.")
    dataset = []

In [3]:
# Convert to DataFrame for easier analysis
df = pd.DataFrame(dataset)

# Display basic statistics and first few rows
print(f"Dataset shape: {df.shape}")
print("\nColumn statistics:")
for col in df.columns:
    print(f"  - {col}: {df[col].nunique()} unique values")
    
df.head()

## 1. Dataset Statistics

Analyzing basic statistics about the dataset.

In [4]:
def analyze_dataset_stats(df):
    """Analyze and visualize basic dataset statistics."""
    # Add length columns
    df['input_length'] = df['input'].apply(lambda x: len(x.split()))
    df['output_length'] = df['output'].apply(lambda x: len(x.split()))
    
    # Instruction distribution
    instruction_counts = df['instruction'].value_counts()
    
    # Plot instruction distribution
    plt.figure(figsize=(12, 8))
    sns.barplot(x=instruction_counts.values, y=instruction_counts.index)
    plt.title('Instruction Template Distribution')
    plt.xlabel('Count')
    plt.tight_layout()
    plt.savefig('visualization_results/01_dataset_stats_instruction_distribution.jpeg')
    plt.close()
    
    # Plot input/output length distributions
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    sns.histplot(df['input_length'], bins=50, kde=True, ax=ax1)
    ax1.set_title('Patient Query Length (words)')
    ax1.set_xlabel('Number of words')
    ax1.axvline(df['input_length'].median(), color='r', linestyle='--', label=f'Median: {df["input_length"].median():.1f}')
    ax1.axvline(df['input_length'].mean(), color='g', linestyle='-', label=f'Mean: {df["input_length"].mean():.1f}')
    ax1.legend()
    
    sns.histplot(df['output_length'], bins=50, kde=True, ax=ax2)
    ax2.set_title('Doctor Response Length (words)')
    ax2.set_xlabel('Number of words')
    ax2.axvline(df['output_length'].median(), color='r', linestyle='--', label=f'Median: {df["output_length"].median():.1f}')
    ax2.axvline(df['output_length'].mean(), color='g', linestyle='-', label=f'Mean: {df["output_length"].mean():.1f}')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('visualization_results/01_dataset_stats_length_distributions.jpeg')
    plt.close()
    
    # Plot length scatter plot
    plt.figure(figsize=(10, 8))
    sns.scatterplot(x='input_length', y='output_length', data=df, alpha=0.3)
    plt.title('Query Length vs Response Length')
    plt.xlabel('Query Length (words)')
    plt.ylabel('Response Length (words)')
    plt.savefig('visualization_results/01_dataset_stats_length_correlation.jpeg')
    plt.close()
    
    # Print statistics summary
    print("\nDataset Statistics:")
    print(f"Total examples: {len(df):,}")
    print(f"Unique instructions: {df['instruction'].nunique():,}")
    print("\nQuery Length (words):")
    print(f"  Min: {df['input_length'].min():,}")
    print(f"  Max: {df['input_length'].max():,}")
    print(f"  Mean: {df['input_length'].mean():.1f}")
    print(f"  Median: {df['input_length'].median():,}")
    print("\nResponse Length (words):")
    print(f"  Min: {df['output_length'].min():,}")
    print(f"  Max: {df['output_length'].max():,}")
    print(f"  Mean: {df['output_length'].mean():.1f}")
    print(f"  Median: {df['output_length'].median():,}")

# Call the function to analyze dataset stats
analyze_dataset_stats(df)

## 2. Patient Demographics Analysis

Extracting and analyzing patient demographics from the dataset.

In [5]:
def extract_patient_demographics(df):
    """Extract patient demographics (age, gender) from queries."""
    # Extract age using regex
    age_pattern = r"\b(\d+)\s*(?:years?|yrs?|y\.?o\.?|year\s+old)\b"
    df['patient_age'] = df['input'].apply(lambda x: 
                                         re.search(age_pattern, x, re.IGNORECASE).group(1) 
                                         if re.search(age_pattern, x, re.IGNORECASE) else None)
    
    # Convert age to numeric
    df['patient_age'] = pd.to_numeric(df['patient_age'], errors='coerce')
    
    # Extract gender using keywords
    def extract_gender(text):
        if re.search(r"\b(?:male|man|boy|gentleman|father|husband|son)\b", text, re.IGNORECASE):
            return "Male"
        elif re.search(r"\b(?:female|woman|girl|lady|mother|wife|daughter)\b", text, re.IGNORECASE):
            return "Female"
        else:
            return None
    
    df['patient_gender'] = df['input'].apply(extract_gender)
    
    return df

def analyze_patient_demographics(df):
    """Analyze and visualize patient demographics."""
    # Extract demographics if not already present
    if 'patient_age' not in df.columns or 'patient_gender' not in df.columns:
        df = extract_patient_demographics(df)
    
    # Age distribution
    plt.figure(figsize=(12, 6))
    sns.histplot(df['patient_age'].dropna(), bins=20, kde=True)
    plt.title('Patient Age Distribution')
    plt.xlabel('Age (years)')
    plt.ylabel('Frequency')
    plt.savefig('visualization_results/02_demographics_age_distribution.jpeg')
    plt.close()
    
    # Age statistics
    age_stats = df['patient_age'].describe()
    
    # Age groups
    age_bins = [0, 12, 18, 30, 45, 60, 75, 100]
    age_labels = ['0-12', '13-18', '19-30', '31-45', '46-60', '61-75', '76+'] 
    df['age_group'] = pd.cut(df['patient_age'], bins=age_bins, labels=age_labels)
    
    # Plot age groups
    plt.figure(figsize=(12, 6))
    age_group_counts = df['age_group'].value_counts().sort_index()
    sns.barplot(x=age_group_counts.index, y=age_group_counts.values)
    plt.title('Patient Age Groups')
    plt.xlabel('Age Group')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.savefig('visualization_results/02_demographics_age_groups.jpeg')
    plt.close()
    
    # Gender distribution
    plt.figure(figsize=(8, 6))
    gender_counts = df['patient_gender'].value_counts()
    plt.pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))
    plt.axis('equal')
    plt.title('Patient Gender Distribution')
    plt.savefig('visualization_results/02_demographics_gender_distribution.jpeg')
    plt.close()
    
    # Age by gender
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='patient_gender', y='patient_age', data=df.dropna(subset=['patient_gender', 'patient_age']))
    plt.title('Age Distribution by Gender')
    plt.savefig('visualization_results/02_demographics_age_by_gender.jpeg')
    plt.close()
    
    # Print statistics
    print("\nPatient Demographics:")
    print(f"Age available for {df['patient_age'].count():,} patients ({df['patient_age'].count()/len(df)*100:.1f}%)")
    print(f"Gender available for {df['patient_gender'].count():,} patients ({df['patient_gender'].count()/len(df)*100:.1f}%)")
    print("\nAge Statistics:")
    print(f"  Mean age: {age_stats['mean']:.1f} years")
    print(f"  Median age: {age_stats['50%']:.1f} years")
    print(f"  Min age: {age_stats['min']:.1f} years")
    print(f"  Max age: {age_stats['max']:.1f} years")
    print("\nGender Distribution:")
    for gender, count in gender_counts.items():
        print(f"  {gender}: {count:,} ({count/len(df)*100:.1f}%)")

# Call the function to analyze patient demographics
analyze_patient_demographics(df)

## 3. Medical Conditions and Symptoms Analysis

Identifying common medical conditions and symptoms in the dataset.

In [6]:
def extract_medical_entities(df):
    """Extract medical entities using spaCy and medical term lists."""
    # Load spaCy model
    try:
        nlp = spacy.load("en_core_web_sm")
    except:
        print("Installing spaCy model...")
        !python -m spacy download en_core_web_sm
        nlp = spacy.load("en_core_web_sm")
    
    # Common medical condition keywords
    medical_conditions = [
        'diabetes', 'hypertension', 'migraine', 'asthma', 'arthritis',
        'allergy', 'depression', 'anxiety', 'cancer', 'infection',
        'pneumonia', 'bronchitis', 'fracture', 'concussion', 'stroke',
        'heart attack', 'fibromyalgia', 'hyperthyroidism', 'hypothyroidism',
        'copd', 'gerd', 'ulcer', 'kidney stone', 'hernia', 'cholesterol',
        'anemia', 'appendicitis', 'autism', 'alzheimer', 'parkinson',
        'bipolar', 'schizophrenia', 'epilepsy', 'glaucoma', 'cataract',
        'osteoporosis', 'psoriasis', 'eczema', 'dermatitis', 'acne',
        'hiv', 'aids', 'hepatitis', 'cirrhosis', 'tuberculosis',
        'ms', 'multiple sclerosis', 'lupus', 'fibroid', 'endometriosis',
        'pcos', 'thyroid', 'seizure', 'flu', 'influenza', 'covid',
        'headache', 'dizziness', 'vertigo', 'insomnia', 'fatigue',
        'fever', 'cough', 'diarrhea', 'constipation', 'vomiting',
        'nausea', 'pain', 'swelling', 'inflammation', 'rash',
        'bleeding', 'bruising', 'numbness', 'tingling', 'weakness'
    ]
    
    # Create a pattern for efficient matching
    condition_pattern = r'\b(' + '|'.join(medical_conditions) + r')\b'
    
    # Extract conditions from input
    def find_conditions(text):
        matches = re.findall(condition_pattern, text.lower())
        return list(set(matches))
    
    # Extract diagnoses from output
    def extract_diagnosis(text):
        diagnosis_match = re.search(r'diagnosis:(.+?)\n\n', text, re.IGNORECASE)
        if diagnosis_match:
            return diagnosis_match.group(1).strip()
        return ""
    
    # Extract treatment from output
    def extract_treatment(text):
        treatment_match = re.search(r'treatment plan:(.+)', text, re.IGNORECASE | re.DOTALL)
        if treatment_match:
            return treatment_match.group(1).strip()
        return ""
    
    # Apply extraction functions
    print("Extracting medical conditions from queries...")
    df['conditions_mentioned'] = df['input'].apply(find_conditions)
    print("Extracting diagnoses from responses...")
    df['diagnosis'] = df['output'].apply(extract_diagnosis)
    print("Extracting treatments from responses...")
    df['treatment'] = df['output'].apply(extract_treatment)
    
    return df

def analyze_medical_conditions(df):
    """Analyze and visualize medical conditions and treatments."""
    # Extract medical entities if not already present
    if 'conditions_mentioned' not in df.columns:
        df = extract_medical_entities(df)
    
    # Count mentioned conditions
    condition_counts = Counter()
    for conditions in df['conditions_mentioned']:
        condition_counts.update(conditions)
    
    # Plot top conditions mentioned
    top_conditions = pd.DataFrame(condition_counts.most_common(20), columns=['Condition', 'Count'])
    plt.figure(figsize=(12, 8))
    sns.barplot(y='Condition', x='Count', data=top_conditions)
    plt.title('Top 20 Medical Conditions Mentioned in Patient Queries')
    plt.tight_layout()
    plt.savefig('visualization_results/03_conditions_top_mentioned.jpeg')
    plt.close()
    
    # Create wordcloud for diagnoses
    diagnosis_text = ' '.join(df['diagnosis'].dropna())
    wordcloud = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(diagnosis_text)
    plt.figure(figsize=(16, 8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Common Words in Diagnoses')
    plt.savefig('visualization_results/03_conditions_diagnosis_wordcloud.jpeg')
    plt.close()
    
    # Create wordcloud for treatments
    treatment_text = ' '.join(df['treatment'].dropna())
    wordcloud = WordCloud(width=800, height=400, background_color='white', max_words=100).generate(treatment_text)
    plt.figure(figsize=(16, 8))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Common Words in Treatment Plans')
    plt.savefig('visualization_results/03_conditions_treatment_wordcloud.jpeg')
    plt.close()
    
    # Statistics about conditions per patient
    df['condition_count'] = df['conditions_mentioned'].apply(len)
    plt.figure(figsize=(10, 6))
    sns.countplot(x='condition_count', data=df[df['condition_count'] <= 10])
    plt.title('Number of Medical Conditions Mentioned per Patient')
    plt.xlabel('Number of Conditions')
    plt.ylabel('Number of Patients')
    plt.savefig('visualization_results/03_conditions_per_patient.jpeg')
    plt.close()
    
    # Print statistics
    print("\nMedical Conditions Analysis:")
    print(f"Total unique conditions identified: {len(condition_counts):,}")
    print(f"Average conditions mentioned per patient: {df['condition_count'].mean():.2f}")
    print("\nTop 10 Medical Conditions:")
    for condition, count in condition_counts.most_common(10):
        print(f"  {condition}: {count:,} mentions")

# Call the function to analyze medical conditions
analyze_medical_conditions(df)

## 4. Medical Specialties Analysis

Identifying medical specialties from the dataset.

In [7]:
def analyze_medical_specialties(df):
    """Identify and analyze potential medical specialties in the dataset using NLP clustering."""
    # TO OPTIMIZE: This function now saves results to files instead of displaying them
    print("Processing medical specialties analysis...")
    
    # Initialize a dictionary to map specialties
    specialty_map = {
        'cardiology': ['heart', 'cardiac', 'chest pain', 'palpitation', 'hypertension', 'bp', 'blood pressure'],
        'dermatology': ['skin', 'rash', 'acne', 'eczema', 'dermatitis', 'psoriasis', 'mole'],
        'gastroenterology': ['stomach', 'abdomen', 'digestive', 'gastric', 'bowel', 'diarrhea', 'constipation'],
        'neurology': ['brain', 'headache', 'migraine', 'seizure', 'epilepsy', 'numbness', 'stroke'],
        'orthopedics': ['bone', 'joint', 'fracture', 'arthritis', 'knee', 'back pain', 'spine'],
        'pediatrics': ['child', 'infant', 'baby', 'toddler', 'childhood', 'newborn'],
        'psychiatry': ['anxiety', 'depression', 'stress', 'bipolar', 'schizophrenia', 'mental health'],
        'obstetrics_gynecology': ['pregnancy', 'ovary', 'uterus', 'menstrual', 'period', 'pcos', 'obgyn'],
        'ent': ['ear', 'nose', 'throat', 'sinus', 'hearing', 'tonsil', 'snoring'],
        'ophthalmology': ['eye', 'vision', 'glasses', 'cataract', 'glaucoma', 'blindness'],
        'urology': ['urinary', 'bladder', 'kidney', 'prostate', 'uti', 'urination'],
        'endocrinology': ['thyroid', 'diabetes', 'hormone', 'insulin', 'glucose'],
        'dentistry': ['tooth', 'teeth', 'dental', 'gum', 'cavity', 'dentist']
    }
    
    # Function to identify specialties in text
    def identify_specialties(text):
        text = text.lower()
        specialties = []
        for specialty, keywords in specialty_map.items():
            for keyword in keywords:
                if keyword in text:
                    specialties.append(specialty)
                    break
        return specialties if specialties else ['general']
    
    # Apply specialty identification
    print("Identifying medical specialties in queries...")
    df['specialties'] = df['input'].apply(identify_specialties)
    
    # Count specialties
    specialty_counts = Counter()
    for specialties_list in df['specialties']:
        specialty_counts.update(specialties_list)
    
    # Plot specialty distribution
    specialty_df = pd.DataFrame(specialty_counts.most_common(), columns=['Specialty', 'Count'])
    plt.figure(figsize=(12, 8))
    sns.barplot(y='Specialty', x='Count', data=specialty_df)
    plt.title('Medical Specialties Distribution')
    plt.tight_layout()
    plt.savefig('visualization_results/04_specialties_distribution.jpeg')
    plt.close()
    
    # Create a dataframe with one row per specialty for length analysis
    specialty_data = []
    for _, row in df.iterrows():
        for specialty in row['specialties']:
            specialty_data.append({
                'specialty': specialty,
                'input_length': row['input_length'],
                'output_length': row['output_length']
            })
    
    specialty_df = pd.DataFrame(specialty_data)
    
    # Plot average query length by specialty
    specialty_length = specialty_df.groupby('specialty').agg({'input_length': 'mean', 'output_length': 'mean'}).reset_index()
    specialty_length = specialty_length.sort_values('input_length', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(y='specialty', x='input_length', data=specialty_length)
    plt.title('Average Query Length by Medical Specialty')
    plt.xlabel('Average Number of Words')
    plt.savefig('visualization_results/04_specialties_query_length.jpeg')
    plt.close()
    
    # Plot average response length by specialty
    specialty_length = specialty_length.sort_values('output_length', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(y='specialty', x='output_length', data=specialty_length)
    plt.title('Average Response Length by Medical Specialty')
    plt.xlabel('Average Number of Words')
    plt.savefig('visualization_results/04_specialties_response_length.jpeg')
    plt.close()
    
    # Specialty by patient age
    if 'patient_age' in df.columns and 'age_group' in df.columns:
        # Create data for specialty by age group
        specialty_age_data = []
        for _, row in df.iterrows():
            if pd.notna(row['age_group']):
                for specialty in row['specialties']:
                    specialty_age_data.append({
                        'specialty': specialty,
                        'age_group': row['age_group']
                    })
        
        specialty_age_df = pd.DataFrame(specialty_age_data)
        
        # Create a pivot table for heatmap
        specialty_age_pivot = pd.crosstab(specialty_age_df['specialty'], specialty_age_df['age_group'], normalize='index')
        
        # Plot heatmap
        plt.figure(figsize=(14, 10))
        sns.heatmap(specialty_age_pivot, cmap='YlGnBu', annot=True, fmt='.2%')
        plt.title('Medical Specialties by Patient Age Group')
        plt.savefig('visualization_results/04_specialties_by_age.jpeg')
        plt.close()
    
    # Specialty by patient gender
    if 'patient_gender' in df.columns:
        # Create data for specialty by gender
        specialty_gender_data = []
        for _, row in df.iterrows():
            if pd.notna(row['patient_gender']):
                for specialty in row['specialties']:
                    specialty_gender_data.append({
                        'specialty': specialty,
                        'gender': row['patient_gender']
                    })
        
        specialty_gender_df = pd.DataFrame(specialty_gender_data)
        
        # Create a pivot table for heatmap
        specialty_gender_pivot = pd.crosstab(specialty_gender_df['specialty'], specialty_gender_df['gender'], normalize='index')
        
        # Plot heatmap
        plt.figure(figsize=(10, 8))
        sns.heatmap(specialty_gender_pivot, cmap='coolwarm', annot=True, fmt='.2%')
        plt.title('Medical Specialties by Patient Gender')
        plt.savefig('visualization_results/04_specialties_by_gender.jpeg')
        plt.close()
    
    # Print statistics to notebook
    print("\nMedical Specialties Analysis:")
    print(f"Total specialties identified: {len(specialty_counts):,}")
    print("\nTop Medical Specialties:")
    for specialty, count in specialty_counts.most_common(10):
        print(f"  {specialty}: {count:,} ({count/len(df)*100:.1f}%)")
    
    print("\nSpecialty analysis complete. All visualizations have been saved to 'visualization_results/' directory.")

# Call the function to analyze medical specialties
analyze_medical_specialties(df)

## 5. Text Clustering Analysis

Using NLP techniques to identify clusters of similar medical cases.

In [8]:
def analyze_text_clusters(df, n_clusters=5, sample_size=5000):
    """Cluster medical cases to identify similar groups."""
    print("Performing text clustering analysis...")
    
    # Use a random sample to speed up clustering
    if len(df) > sample_size:
        sample_df = df.sample(sample_size, random_state=42)
    else:
        sample_df = df
    
    # Vectorize input text
    print("Vectorizing input text...")
    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    X = vectorizer.fit_transform(sample_df['input'])
    
    # Perform KMeans clustering
    print(f"Clustering into {n_clusters} groups...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    sample_df['cluster'] = kmeans.fit_predict(X)
    
    # Get top terms for each cluster
    feature_names = vectorizer.get_feature_names_out()
    centroids = kmeans.cluster_centers_
    
    # Function to get top terms
    def get_top_terms(centroid, n=10):
        indices = centroid.argsort()[-n:][::-1]
        return [feature_names[i] for i in indices]
    
    # Get top terms for each cluster
    cluster_terms = {}
    for i, centroid in enumerate(centroids):
        cluster_terms[i] = get_top_terms(centroid)
    
    # Plot cluster sizes
    plt.figure(figsize=(10, 6))
    cluster_sizes = sample_df['cluster'].value_counts().sort_index()
    cluster_sizes.plot(kind='bar')
    plt.title('Cluster Sizes')
    plt.xlabel('Cluster')
    plt.ylabel('Number of Examples')
    plt.xticks(rotation=0)
    plt.savefig('visualization_results/05_clusters_sizes.jpeg')
    plt.close()
    
    # Visualize top terms for each cluster
    for cluster_id, terms in cluster_terms.items():
        plt.figure(figsize=(12, 4))
        y_pos = range(len(terms))
        plt.barh(y_pos, range(len(terms), 0, -1))
        plt.yticks(y_pos, terms)
        plt.title(f'Top Terms in Cluster {cluster_id}')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.savefig(f'visualization_results/05_clusters_{cluster_id}_terms.jpeg')
        plt.close()
    
    # Plot cluster and specialty relationship if available
    if 'specialties' in sample_df.columns:
        # Create a dataframe with one row per specialty-cluster combination
        cluster_specialty_data = []
        for _, row in sample_df.iterrows():
            for specialty in row['specialties']:
                cluster_specialty_data.append({
                    'cluster': row['cluster'],
                    'specialty': specialty
                })
        
        cluster_specialty_df = pd.DataFrame(cluster_specialty_data)
        
        # Create a pivot table
        cluster_specialty_pivot = pd.crosstab(cluster_specialty_df['specialty'], cluster_specialty_df['cluster'])
        
        # Normalize by cluster
        cluster_specialty_norm = cluster_specialty_pivot.div(cluster_specialty_pivot.sum(axis=0), axis=1)
        
        # Plot heatmap
        plt.figure(figsize=(12, 10))
        sns.heatmap(cluster_specialty_norm, cmap='viridis', annot=True, fmt='.2f')
        plt.title('Relationship Between Clusters and Medical Specialties')
        plt.savefig('visualization_results/05_clusters_specialty_heatmap.jpeg')
        plt.close()
    
    # Print cluster information
    print("\nText Clustering Results:")
    print(f"Number of clusters: {n_clusters}")
    print("\nCluster Sizes:")
    for cluster_id, size in cluster_sizes.items():
        print(f"  Cluster {cluster_id}: {size:,} examples ({size/len(sample_df)*100:.1f}%)")
    
    print("\nTop Terms for Each Cluster:")
    for cluster_id, terms in cluster_terms.items():
        print(f"  Cluster {cluster_id}: {', '.join(terms)}")
    
    print("\nClustering analysis complete. All visualizations have been saved to 'visualization_results/' directory.")

# Call the function to analyze text clusters
analyze_text_clusters(df)

## 6. Dataset Quality Analysis

Analyzing dataset quality and identifying potential issues.

In [9]:
def analyze_dataset_quality(df):
    """Analyze dataset quality and identify potential issues."""
    print("Performing dataset quality analysis...")
    
    # Check for missing or empty values
    missing_values = {}
    for col in ['instruction', 'input', 'output']:
        missing_values[col] = {
            'null': df[col].isnull().sum(),
            'empty': (df[col] == '').sum(),
            'whitespace_only': (df[col].str.strip() == '').sum() if df[col].dtype == 'object' else 0
        }
    
    # Check for very short inputs/outputs
    short_inputs = (df['input_length'] < 5).sum()
    short_outputs = (df['output_length'] < 10).sum()
    
    # Check for very long inputs/outputs
    long_inputs = (df['input_length'] > 500).sum()
    long_outputs = (df['output_length'] > 1000).sum()
    
    # Check for duplicates
    duplicate_inputs = df.duplicated(subset=['input']).sum()
    duplicate_outputs = df.duplicated(subset=['output']).sum()
    duplicate_rows = df.duplicated().sum()
    
    # Check for generic responses
    generic_patterns = [
        r'consult.*doctor',
        r'see.*physician',
        r'cannot.*diagnose',
        r'need.*more information',
        r'without.*examination'
    ]
    
    generic_responses = 0
    for pattern in generic_patterns:
        generic_responses += df['output'].str.contains(pattern, case=False, regex=True).sum()
    
    # Plot distribution of issues
    issues = {
        'Missing instructions': missing_values['instruction']['null'] + missing_values['instruction']['empty'],
        'Missing inputs': missing_values['input']['null'] + missing_values['input']['empty'],
        'Missing outputs': missing_values['output']['null'] + missing_values['output']['empty'],
        'Very short inputs': short_inputs,
        'Very short outputs': short_outputs,
        'Very long inputs': long_inputs,
        'Very long outputs': long_outputs,
        'Duplicate inputs': duplicate_inputs,
        'Duplicate outputs': duplicate_outputs,
        'Duplicate rows': duplicate_rows,
        'Generic responses': generic_responses
    }
    
    # Create issues dataframe
    issues_df = pd.DataFrame({
        'Issue': list(issues.keys()),
        'Count': list(issues.values()),
        'Percentage': [count/len(df)*100 for count in issues.values()]
    })
    
    issues_df = issues_df.sort_values('Count', ascending=False)
    
    # Plot issues
    plt.figure(figsize=(12, 8))
    sns.barplot(y='Issue', x='Percentage', data=issues_df)
    plt.title('Dataset Quality Issues (% of Examples)')
    plt.xlabel('Percentage of Examples')
    plt.tight_layout()
    plt.savefig('visualization_results/06_quality_issues.jpeg')
    plt.close()
    
    # Calculate quality score (simple metric)
    total_issues = sum(issues.values())
    quality_score = max(0, 100 - (total_issues / len(df) * 100))
    
    # Print quality statistics
    print("\nDataset Quality Analysis:")
    print(f"Overall quality score: {quality_score:.1f}/100")
    print("\nIdentified Issues:")
    for issue, count in issues.items():
        if count > 0:
            print(f"  {issue}: {count:,} examples ({count/len(df)*100:.2f}%)")
    
    print("\nQuality analysis complete. Visualization saved to 'visualization_results/' directory.")

# Call the function to analyze dataset quality
analyze_dataset_quality(df)

## 7. Summary and Recommendations

Summarizing findings and providing recommendations for fine-tuning.

In [10]:
def generate_summary():
    """Generate summary and recommendations based on all analyses."""
    summary = """
    # HealthCareMagic-100K Dataset Analysis Summary
    
    ## Key Findings
    
    1. **Dataset Composition**:
       - The dataset contains patient-doctor conversations focused on medical diagnosis and treatment.
       - There are variations in query and response lengths, with most queries being concise and responses more detailed.
    
    2. **Patient Demographics**:
       - Age and gender information can be extracted from the queries, providing valuable context.
       - The dataset covers a wide range of age groups and includes both male and female patients.
    
    3. **Medical Conditions**:
       - Common medical conditions include pain, headache, fever, and anxiety.
       - Responses typically include both diagnosis and treatment information.
    
    4. **Medical Specialties**:
       - The dataset spans multiple medical specialties, with some specialties being more represented than others.
       - Different specialties have varying response patterns and lengths.
    
    5. **Text Clustering**:
       - Natural clusters in the data align with medical specialties and common conditions.
       - Clustering can help identify underlying patterns and group similar cases.
    
    6. **Dataset Quality**:
       - Some quality issues exist, including generic responses and potential duplicates.
       - Overall, the dataset appears suitable for fine-tuning with proper preprocessing.
    
    ## Recommendations for Fine-Tuning
    
    1. **Data Filtering**:
       - Remove examples with empty or very short inputs/outputs.
       - Filter out duplicate entries to prevent overfitting.
       - Consider removing very generic responses that don't provide meaningful information.
    
    2. **Data Augmentation**:
       - Balance underrepresented specialties with data augmentation techniques.
       - Consider generating additional examples for rare medical conditions.
    
    3. **Structured Output Format**:
       - Maintain the "Diagnosis" and "Treatment Plan" structure for consistent outputs.
       - Ensure outputs contain comprehensive and actionable information.
    
    4. **Instruction Variation**:
       - Use a variety of instruction templates to improve model adaptability.
       - Include specialty-specific instructions for better domain adaptation.
    
    5. **Evaluation Strategy**:
       - Develop specialty-specific evaluation metrics.
       - Include factual accuracy and completeness checks in the evaluation.
       - Ensure safety evaluations to prevent harmful medical advice.
    
    6. **Training Approach**:
       - Consider a two-stage fine-tuning: first on general medical knowledge, then on specific specialties.
       - Use a lower learning rate to preserve medical knowledge from the base model.
       - Monitor performance on different specialties during training.
    
    All visualizations and detailed analyses have been saved to the 'visualization_results/' directory.
    """
    
    print(summary)
    
    # Also save summary to a file
    with open('visualization_results/07_summary_and_recommendations.md', 'w') as f:
        f.write(summary)
    
    print("\nSummary and recommendations have been saved to 'visualization_results/07_summary_and_recommendations.md'")

# Generate summary and recommendations
generate_summary()

In [11]:
print("\nDataset exploration complete! All visualizations and analyses have been saved to the 'visualization_results/' directory.")
print(f"Total files generated: {len(os.listdir('visualization_results'))}")
print("\nThese files can be used for reporting and for informing the fine-tuning process.")